In [1]:
from dotenv import load_dotenv; load_dotenv()
from fortyguard import FortyGuardClient
from datetime import date, timedelta
import json, pathlib, time

client = FortyGuardClient()

WEST, SOUTH, EAST, NORTH = -121.93, 37.31, -121.86, 37.36
AOI = {"type": "FeatureCollection", "features": [{
    "type": "Feature", "properties": {},
    "geometry": {"type": "Polygon", "coordinates": [[
        [WEST, SOUTH], [EAST, SOUTH], [EAST, NORTH], [WEST, NORTH], [WEST, SOUTH]
    ]]}}]}

START = date(2024, 7, 8)
DAYS  = 21
GRAN  = 100

CACHE = pathlib.Path("data/heatmaps"); CACHE.mkdir(parents=True, exist_ok=True)

for i in range(DAYS):
    d = (START + timedelta(days=i)).isoformat()
    p = CACHE / f"day_{d}.json"
    if p.exists():
        print(f"{d} cached, skip"); continue
    try:
        r = client.create_heatmap(
            polygon_aoi=AOI,
            start_date=d,
            filter_type=3,        # 3 = whole day, 24h
            granularity=GRAN,
        )
        p.write_text(json.dumps(r))
        print(f"{d} ok")
    except Exception as e:
        print(f"{d} FAILED: {e}")
        time.sleep(5)

Submitted -> activity_id=611c464b-5a2d-4f23-8aa9-9d3d4e3f0f79
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
2024-07-08 ok
Submitted -> act

In [2]:
import json, pathlib
for p in sorted(pathlib.Path("data/heatmaps").glob("day_*.json")):
    feats = json.loads(p.read_text())["result"]["map_data"]["features"]
    peaks = [f["properties"]["max_temperature"] for f in feats]
    print(p.stem[4:], f"median {sorted(peaks)[len(peaks)//2]:.1f}")

2024-07-08 median 32.3
2024-07-09 median 30.0
2024-07-10 median 35.3
2024-07-11 median 38.2
2024-07-12 median 35.7
2024-07-13 median 31.9
2024-07-14 median 30.1
2024-07-15 median 27.3
2024-07-16 median 28.3
2024-07-17 median 27.4
2024-07-18 median 33.6
2024-07-19 median 37.0
2024-07-20 median 32.6
2024-07-21 median 29.1
2024-07-22 median 35.4
2024-07-23 median 37.5
2024-07-24 median 35.4
2024-07-25 median 34.4
2024-07-26 median 31.0
2024-07-27 median 22.6
2024-07-28 median 30.9


In [ ]:
import pandas as pd
from shapely.geometry import shape

rows = []
for p in sorted(pathlib.Path("data/heatmaps").glob("day_*.json")):
    r = json.loads(p.read_text())
    day = p.stem.replace("day_", "")
    for f in r["result"]["map_data"]["features"]:
        pr = f["properties"]
        c = shape(f["geometry"]).centroid
        rows.append({
            "date": day,
            "tile_id": pr.get("tile_id"),
            "lon": round(c.x, 6), "lat": round(c.y, 6),
            "avg_c": pr.get("average_temperature"),
            "max_c": pr.get("max_temperature"),
        })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
# df.to_parquet("data/panel.parquet")

# print(f"rows: {len(df)}   tiles: {df.tile_id.nunique()}   days: {df.date.nunique()}")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [4]:
df.to_parquet("data/panel.parquet")

print(f"rows: {len(df)}   tiles: {df.tile_id.nunique()}   days: {df.date.nunique()}")

ArrowKeyError: No type extension with name arrow.py_extension_type found

In [5]:
df.to_csv("data/panel.csv", index=False)
print(f"rows: {len(df)}   tiles: {df.tile_id.nunique()}   days: {df.date.nunique()}")

rows: 70917   tiles: 3377   days: 21


In [6]:
df = pd.read_csv("data/panel.csv", parse_dates=["date"])